V1

In [ ]:
# --- Imports ---
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import io
from heapq import heappop, heappush

# --- Upload-Widget ---
uploader = widgets.FileUpload(accept='.png, .jpg, .jpeg', multiple=False)
display(uploader)

# --- Funktion zum Laden des hochgeladenen Bildes ---
def load_uploaded_image(uploader):
    if not uploader.value:
        print("Kein Bild hochgeladen!")
        return None
    uploaded_file = next(iter(uploader.value.values()))
    content = uploaded_file['content']
    img = Image.open(io.BytesIO(content)).convert('L')  # Graustufen
    img_cv = np.array(img)
    return img_cv

# --- Warten bis du das Bild hochlädst, dann ausführen: ---
# maze_img = load_uploaded_image(uploader)

# --- Maze Processing & Solver ---
def solve_maze(maze_img, grid_size=20):
    # Binarisieren
    _, binary_maze = cv2.threshold(maze_img, 128, 1, cv2.THRESH_BINARY_INV)

    # Grobes Grid erstellen
    cell_size_x = binary_maze.shape[0] // grid_size
    cell_size_y = binary_maze.shape[1] // grid_size

    coarse_maze = np.zeros((grid_size, grid_size), dtype=np.int32)

    for i in range(grid_size):
        for j in range(grid_size):
            cell = binary_maze[i*cell_size_x:(i+1)*cell_size_x, j*cell_size_y:(j+1)*cell_size_y]
            if np.mean(cell) > 0.2:
                coarse_maze[i, j] = 1

    # Hilfsfunktionen
    def get_neighbors(pos, grid):
        neighbors = []
        x, y = pos
        for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid.shape[0] and 0 <= ny < grid.shape[1]:
                if grid[nx, ny] == 0:
                    neighbors.append((nx, ny))
        return neighbors

    def heuristic(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def astar(grid, start, goal):
        open_set = []
        heappush(open_set, (0 + heuristic(start, goal), 0, start))
        came_from = {}
        g_score = {start: 0}

        while open_set:
            _, cost, current = heappop(open_set)

            if current == goal:
                path = []
                while current in came_from:
                    path.append(current)
                    current = came_from[current]
                path.append(start)
                path.reverse()
                return path

            for neighbor in get_neighbors(current, grid):
                tentative_g = g_score[current] + 1
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_score = tentative_g + heuristic(neighbor, goal)
                    heappush(open_set, (f_score, tentative_g, neighbor))
        return None

    def find_first_free(grid):
        for x in range(grid.shape[0]):
            for y in range(grid.shape[1]):
                if grid[x, y] == 0:
                    return (x, y)
        return None

    def find_last_free(grid):
        for x in reversed(range(grid.shape[0])):
            for y in reversed(range(grid.shape[1])):
                if grid[x, y] == 0:
                    return (x, y)
        return None

    # Start & Ziel finden
    start = find_first_free(coarse_maze)
    goal = find_last_free(coarse_maze)

    # A* Pathfinding
    path = astar(coarse_maze, start, goal)

    if path is None:
        print("Kein Pfad gefunden!")
        return

    # Lösung einzeichnen
    solved_img = cv2.cvtColor(maze_img, cv2.COLOR_GRAY2BGR)

    for (grid_x, grid_y) in path:
        px = grid_x * cell_size_x + cell_size_x // 2
        py = grid_y * cell_size_y + cell_size_y // 2
        cv2.circle(solved_img, (py, px), radius=cell_size_x//4, color=(0,0,255), thickness=-1)

    # Ergebnis anzeigen
    plt.figure(figsize=(8,8))
    plt.imshow(solved_img)
    plt.title('Maze mit Lösungsweg')
    plt.axis('off')
    plt.show()

# --- NACHDEM DU EIN BILD HOCHGELADEN HAST: ---
maze_img = load_uploaded_image(uploader)
solve_maze(maze_img)


FileUpload(value=(), accept='.png, .jpg, .jpeg', description='Upload')